In [85]:
##import package
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score

In [36]:
###read data
##initial data
pool_data = pd.read_csv('data/MushroomDataset/secondary_data.csv', sep=';')

In [86]:
# helper function
def d_optimal_initialization(X, n_init=200):
    """
    D-Optimal Design: select points that maximize feature space coverage
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    n_samples = X_scaled.shape[0]
    selected = []
    remaining = set(range(n_samples))
    
    # Start with point furthest from origin
    norms = np.linalg.norm(X_scaled, axis=1)
    first_idx = np.argmax(norms)
    selected.append(first_idx)
    remaining.remove(first_idx)
    
    for i in range(n_init - 1):
        best_idx = None
        best_det = -np.inf
        
        X_selected = X_scaled[selected]
        
        # Sample 1000 candidates for efficiency
        candidates = list(remaining)
        if len(candidates) > 1000:
            candidates = np.random.choice(candidates, 1000, replace=False)
        
        for idx in candidates:
            X_candidate = np.vstack([X_selected, X_scaled[idx]])
            XtX = X_candidate.T @ X_candidate
            _, logdet = np.linalg.slogdet(XtX)
            
            if logdet > best_det:
                best_det = logdet
                best_idx = idx
        
        selected.append(best_idx)
        remaining.remove(best_idx)
     
    return np.array(selected)

def random_sampling(X_pool, n_samples=50):
    """Randomly select n_samples from pool"""
    indices = np.random.choice(len(X_pool), n_samples, replace=False)
    return indices

def Active_learning(model, X_initial, y_initial, X_pool, y_pool,
                         query_strategy, batch_size=50, target_accuracy=0.95, cv=5):
    """
    Active Learning loop using cross-validation accuracy
    
    Stopping criteria:
    1. Target CV accuracy reached
    2. Pool exhausted
    """
    # Copy data
    X_labeled = X_initial.copy().reset_index(drop=True)
    y_labeled = y_initial.copy().reset_index(drop=True)
    X_unlabeled = X_pool.copy().reset_index(drop=True)
    y_unlabeled = y_pool.copy().reset_index(drop=True)
    
    results = []
    round_num = 0
    
    while len(X_unlabeled) >= batch_size:
        # Cross-validation accuracy
        cv_scores = cross_val_score(model, X_labeled, y_labeled, cv=cv, scoring='accuracy')
        cv_accuracy = cv_scores.mean()
        cv_std = cv_scores.std()
        
        results.append({
            'labels': len(y_labeled), 
            'cv_accuracy': cv_accuracy,
            'cv_std': cv_std
        })
                
        # Check stopping criteria
        if cv_accuracy >= target_accuracy:
            print(f" Target accuracy {target_accuracy:.0%} reached!")
            break
        
        # Query next batch
        query_indices = query_strategy(X_unlabeled, batch_size)
        
        # Move from pool to labeled set
        X_labeled = pd.concat([X_labeled, X_unlabeled.iloc[query_indices]], ignore_index=True)
        y_labeled = pd.concat([y_labeled, y_unlabeled.iloc[query_indices]], ignore_index=True)
        
        # Remove from pool
        X_unlabeled = X_unlabeled.drop(query_indices).reset_index(drop=True)
        y_unlabeled = y_unlabeled.drop(query_indices).reset_index(drop=True)
        
    
    else:
        print("Pool exhausted")
    
    return results

In [80]:
##preprocessing
## Step 1: treat NaN value as missing type
pool_data = pool_data.fillna('missing')

# Step 2: Separate target and features
y = pool_data['class'].map({'e': 0, 'p': 1}) 
X = pool_data.drop(columns=['class'])

# Step 3:one-hot encoding
encoders = {}
numeric_cols = ['cap-diameter', 'stem-height', 'stem-width']
categorical_cols = [col for col in X.columns if col not in numeric_cols]

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

##initial data, then rest to be oracle and query data(random setting)
X_initial_random, X_pool_random, y_initial_random, y_pool_random = train_test_split(
    X, y, 
    train_size=200, 
    random_state=42,
    stratify=y 
)

# initial data via D-opt
D_opt_indices = d_optimal_initialization(X.values, n_init=200)

X_initial_D_opt = X.iloc[D_opt_indices].reset_index(drop=True)
y_initial_D_opt = y.iloc[D_opt_indices].reset_index(drop=True)
pool_indices = [i for i in range(len(X)) if i not in D_opt_indices]
X_pool_D_opt = X.iloc[pool_indices].reset_index(drop=True)
y_pool_D_opt = y.iloc[pool_indices].reset_index(drop=True)


In [84]:
## define base learning 
LR = LogisticRegression(max_iter=1000, random_state=42)

RF = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
MLP = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=500,
    random_state=42
)

In [90]:
## an example
result = Active_learning(
    model=RF,
    X_initial=X_initial,
    y_initial=y_initial,
    X_pool=X_pool,
    y_pool=y_pool,
    query_strategy=random_sampling,
    batch_size=50,
    target_accuracy=0.97,
    cv=5
)

 Target accuracy 97% reached!
